<img src="https://hilpisch.com/tpq_logo_bic.png" width="20%" align="right">


# Python for Finance, 3rd Edition
## Lab 02 · Summarizing Earnings Surprises with LLMs and Python

&copy; Dr. Yves J. Hilpisch<br>
AI-supported by GPT 5.x<br>
The Python Quants GmbH | https://tpq.io<br>
https://hilpisch.com | https://linktr.ee/dyjh


## Notebook Goals
This notebook mirrors the lab examples in a Colab-ready format so that you can
run, tweak, and extend them interactively.


### How to Use This Notebook
- Run the cells top to bottom the first time to create all variables.
- Use additional cells for your own experiments or GenAI-assisted
  refactorings.
- Refer back to the lab text for detailed explanations and context.


## Project Setup
Set the project root so local data files, helper modules, and figure scripts
resolve correctly from `notebooks/labs/`.


In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().parents[1]
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

loaded_code = sys.modules.get("code")
if loaded_code is not None and not hasattr(loaded_code, "__path__"):
    del sys.modules["code"]

PROJECT_ROOT

The first step is to store a few text items in a tabular file with
stable metadata.


In [ ]:
import pandas as pd

In [ ]:
df = pd.read_csv("../../data/earnings_text_samples.csv")

In [ ]:
df[["ticker", "report_date", "source"]]

## Building the Schema in Python
Before calling any model, it helps to make the target structure explicit in
code.


In [ ]:
SUMMARY_FIELDS = [
    "ticker",
    "report_date",
    "summary",
    "surprise_direction",
    "guidance_change",
    "main_positive_driver",
    "main_negative_driver",
    "risk_flags",
    "confidence_note",
]

In [ ]:
SUMMARY_FIELDS

## Constructing Prompts Programmatically
Instead of composing each prompt by hand, you can build it directly from a row
in the raw text table.


## Parsing Structured Responses
Once you ask for JSON, the next task is to parse and normalize the model
output into a table.


In [ ]:
import json

In [ ]:
sample_response = """
{
  "ticker": "AAPL",
  "report_date": "2026-01-29",
  "summary": "Revenue beat expectations, driven by services and wearables.",
  "surprise_direction": "positive",
  "guidance_change": "unclear",
  "main_positive_driver": "services and wearables growth",
  "main_negative_driver": "foreign-exchange headwinds and weaker demand",
  "risk_flags": ["foreign_exchange", "greater_china_demand"],
  "confidence_note": "fairly explicit"
}
"""

In [ ]:
record = json.loads(sample_response)

In [ ]:
keys = ("ticker", "surprise_direction", "guidance_change")

In [ ]:
{key: record[key] for key in keys}

## From Records to a Review Table
Once each response has been parsed into a dictionary, building a review table
is straightforward.


In [ ]:
summary_df = pd.read_json(
    "../../data/earnings_summary_samples.json"
)

In [ ]:
summary_df[[
    "ticker",
    "surprise_direction",
    "guidance_change"
]].to_dict("records")

## Validating the Output Schema
Before interpreting the summaries, you should make sure the expected fields
are present and populated.


In [ ]:
required = set(SUMMARY_FIELDS)

In [ ]:
missing_cols = sorted(required - set(summary_df.columns))

In [ ]:
null_counts = summary_df[SUMMARY_FIELDS].isna().sum().to_dict()

In [ ]:
{"missing_columns": missing_cols, "null_counts": null_counts}

The most useful review step is to inspect the source text next to the
extracted fields.


In [ ]:
joined = df.merge(
    summary_df, on=["ticker", "report_date"], how="left"
)

In [ ]:
joined[[
    "ticker",
    "text",
    "surprise_direction",
    "main_positive_driver",
    "main_negative_driver"
]].loc[[0, 2]].to_dict("records")

<img src="https://hilpisch.com/tpq_logo_bic.png" width="20%" align="right">
